# Day 16. Residual Connection / LayerNorm

**오늘의 목표**
- Residual Connection `x + f(x)`가 무엇이고, 왜 깊은 모델에 꼭 필요한지 실험으로 확인한다
- "Gradient 고속도로"와 "Residual Stream"이라는 두 가지 관점으로 설명할 수 있다
- LayerNorm이 **Token 하나의 벡터(C 축)** 를 평균 0, 분산 1로 맞추는 과정을 손으로 계산할 수 있다
- `nn.LayerNorm`을 직접 구현하고, 학습 가능한 `weight`/`bias`가 왜 있는지 안다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

# Part 1. Residual Connection

## 1. 아이디어: "원래 것은 두고, 바뀐 만큼만 더한다"

```text
x
│
├──────────────┐
↓              │
Attention      │      out = x + Attention(x)
↓              │
+ ←────────────┘
```

일반적인 층은 입력을 **통째로 새 값으로 바꿉니다**: `out = f(x)`

Residual Connection은 입력을 **그대로 남겨두고, f가 계산한 변화량만 더합니다**: `out = x + f(x)`

비유하자면 문서를 매번 새로 쓰는 게 아니라, 원본 위에 **수정 사항만 덧붙이는** 방식이에요.
f가 쓸모없는 값을 내더라도(예: 0) 최소한 원래 정보 `x`는 그대로 살아남습니다.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
f = nn.Linear(3, 3)

print("x          :", r(x))
print("f(x)       :", r(f(x)))
print("x + f(x)   :", r(x + f(x)))

# f가 아무것도 안 하면(출력 0)?
with torch.no_grad():
    f.weight.zero_()
    f.bias.zero_()
print()
print("f가 0을 내면 → f(x)     :", r(f(x)), " (정보가 전부 사라짐)")
print("f가 0을 내면 → x + f(x) :", r(x + f(x)), " (원래 정보 유지)")

## 2. 실험: 깊게 쌓으면 무슨 일이?

`Linear → tanh` 층을 **30개** 쌓은 모델 두 개를 비교합니다.
- **Plain**: `x = tanh(Linear(x))` 를 30번
- **Residual**: `x = x + tanh(Linear(x))` 를 30번

Loss를 계산하고 `backward()`한 뒤, **각 층의 Weight가 받은 Gradient 크기**를 재봅니다.
Gradient가 0에 가까우면 그 층은 "얼마나 고쳐야 할지 모른다" = **학습이 안 된다**는 뜻이에요. (Day 7)

In [ ]:
depth, width = 30, 16

class DeepNet(nn.Module):
    def __init__(self, residual):
        super().__init__()
        self.residual = residual
        self.layers = nn.ModuleList([nn.Linear(width, width) for _ in range(depth)])

    def forward(self, x):
        for layer in self.layers:
            if self.residual:
                x = x + torch.tanh(layer(x))   # 원래 것 + 변화량
            else:
                x = torch.tanh(layer(x))       # 통째로 바꾸기
        return x

def grad_norms(model):
    torch.manual_seed(1)
    x = torch.randn(64, width)
    target = torch.randn(64, width)
    loss = F.mse_loss(model(x), target)
    model.zero_grad()
    loss.backward()
    return [layer.weight.grad.norm().item() for layer in model.layers]

torch.manual_seed(0)
plain = DeepNet(residual=False)
torch.manual_seed(0)
resid = DeepNet(residual=True)

g_plain = grad_norms(plain)
g_resid = grad_norms(resid)

print(f"{'층':>4s} {'Plain':>12s} {'Residual':>12s}")
for i in [0, 1, 5, 10, 20, 29]:
    print(f"{i+1:>4d} {g_plain[i]:>12.2e} {g_resid[i]:>12.2e}")

`1.23e-05`는 `0.0000123`이라는 뜻입니다. Plain 모델은 **입력 쪽(1층)으로 갈수록 Gradient가 거의 0**이 됩니다.
그래프로 보면 더 분명해요. (세로축은 log10 값: -8이면 0.00000001, 1 내려갈 때마다 10배 작아짐)

In [ ]:
plt.figure(figsize=(8, 4))
import math
plt.plot(range(1, depth + 1), [math.log10(g) for g in g_plain], "o-", label="Plain: x = f(x)")
plt.plot(range(1, depth + 1), [math.log10(g) for g in g_resid], "o-", label="Residual: x = x + f(x)")
plt.xlabel("층 번호 (1 = 입력 쪽, 30 = 출력 쪽)")
plt.ylabel("log10(Gradient 크기)")
plt.title("깊은 모델에서 층별 Gradient 크기", fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

이 현상을 **Gradient 소실(Vanishing Gradient)** 이라고 부릅니다.
Plain 모델의 앞쪽 층들은 Gradient가 너무 작아서 학습이 거의 안 돼요.

### 2-1. 실제로 학습시켜 보면

In [ ]:
torch.manual_seed(0)
X = torch.randn(256, width)
Y = torch.sin(X * 2)          # 아무 비선형 관계

def train(model, steps=600):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for _ in range(steps):
        loss = F.mse_loss(model(X), Y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

torch.manual_seed(0)
loss_plain = train(DeepNet(residual=False))
torch.manual_seed(0)
loss_resid = train(DeepNet(residual=True))

plt.figure(figsize=(8, 4))
plt.plot(loss_plain, label=f"Plain (최종 {loss_plain[-1]:.3f})")
plt.plot(loss_resid, label=f"Residual (최종 {loss_resid[-1]:.3f})")
plt.xlabel("step")
plt.ylabel("Loss")
plt.title("30층 모델 학습 곡선", fontsize=13)
plt.ylim(0, 1.2)          # Residual의 처음 Loss는 아주 커서 위쪽은 잘라서 봅니다
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Plain    : 처음 {loss_plain[0]:.3f} → 마지막 {loss_plain[-1]:.3f}")
print(f"Residual : 처음 {loss_resid[0]:.3f} → 마지막 {loss_resid[-1]:.3f}")

- **Plain**: Loss가 처음부터 거의 **평평**합니다. 앞쪽 층들이 Gradient를 못 받아서 사실상 학습이 멈춰 있어요.
- **Residual**: 처음엔 값이 쌓여서 Loss가 크게 시작하지만(이 문제는 Part 2의 LayerNorm이 해결합니다), **꾸준히 내려가서** Plain보다 낮아집니다.

## 3. 왜 Residual이면 Gradient가 살아남나: "Gradient 고속도로"

미분으로 보면 간단합니다. (Day 7의 연쇄 법칙)

```text
Plain    : out = f(x)       →  d out / dx = f'(x)
Residual : out = x + f(x)   →  d out / dx = 1 + f'(x)
                                            ↑
                                    항상 1이 더해짐!
```

층을 30개 지나면 이 값들이 **30번 곱해집니다**.
- Plain: `f'` 가 0.5 정도면 `0.5³⁰ ≈ 0.000000001` → 사라짐
- Residual: `1 + f'` 에는 **1이라는 지름길**이 항상 있어서, Gradient가 층을 건너뛰어 앞까지 그대로 전달될 수 있음

이 지름길을 흔히 **Gradient 고속도로(gradient highway)** 라고 부릅니다.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

out = x + 0.1 * x ** 2          # f(x) = 0.1x²  →  f'(x) = 0.2x = 0.4
out.backward()
print("x + f(x) 의 Gradient :", round(x.grad.item(), 3), " = 1 + f'(x) = 1 + 0.4")

x.grad = None
out = 0.1 * x ** 2               # Residual 없이
out.backward()
print("f(x) 만의 Gradient   :", round(x.grad.item(), 3), " = f'(x)")

## 4. 또 다른 관점: Residual Stream

Transformer에서는 Residual을 이렇게 상상하면 이해가 쉽습니다.

```text
Embedding ──▶━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━▶ lm_head
                 │   ▲          │   ▲          │   ▲
                 ▼   │          ▼   │          ▼   │
               Attention       FFN         Attention   ...
               (읽고 → 더함)  (읽고 → 더함)  (읽고 → 더함)
```

- 굵은 선 = **Residual Stream**: Token 벡터가 흘러가는 공용 통로
- 각 Attention / FFN은 Stream에서 **읽어서**, 계산한 결과를 Stream에 **더해 넣기만** 합니다
- 아무 층도 Stream을 "덮어쓰지" 않아요

그래서 `x + f(x)`가 가능하려면 **f의 출력 shape가 입력 shape와 정확히 같아야** 합니다.
Day 4의 FFN이 `C → 4C → C`로 **다시 원래 크기로 돌아오는** 이유가 바로 이거예요.

In [ ]:
B, T, C = 2, 4, 32
x = torch.randn(B, T, C)

ffn_ok  = nn.Sequential(nn.Linear(C, 4 * C), nn.GELU(), nn.Linear(4 * C, C))
ffn_bad = nn.Sequential(nn.Linear(C, 4 * C), nn.GELU())      # 돌아오지 않음

print("x + ffn_ok(x) :", tuple((x + ffn_ok(x)).shape))
try:
    x + ffn_bad(x)
except RuntimeError as e:
    print("x + ffn_bad(x): RuntimeError →", str(e)[:80])

# Part 2. Layer Normalization

## 5. 평균, 분산, 정규화 (커리큘럼 4-3)

Token 하나의 벡터가 있다고 합시다.

```text
v = [2, 4, 6, 8]

평균 (mean)     = (2 + 4 + 6 + 8) / 4 = 5
분산 (variance) = 평균에서 얼마나 퍼져 있나
                = ((2-5)² + (4-5)² + (6-5)² + (8-5)²) / 4
                = (9 + 1 + 1 + 9) / 4 = 5
표준편차 (std)  = √분산 = √5 ≈ 2.236
```

**정규화(Normalization)**: 평균을 빼고 표준편차로 나누면 → **평균 0, 분산 1**인 벡터가 됩니다.

```text
(v - 평균) / 표준편차 = [-3, -1, 1, 3] / 2.236 = [-1.342, -0.447, 0.447, 1.342]
```

In [ ]:
v = torch.tensor([2.0, 4.0, 6.0, 8.0])

mean = v.mean()
var  = ((v - mean) ** 2).mean()
std  = var.sqrt()
v_norm = (v - mean) / std

print("평균      :", mean.item())
print("분산      :", var.item())
print("표준편차  :", round(std.item(), 3))
print("정규화    :", r(v_norm))
print("→ 정규화 후 평균:", round(v_norm.mean().item(), 4), " 분산:", round(v_norm.var(unbiased=False).item(), 4))

> 💡 크기가 전혀 다른 벡터도 정규화하면 **같은 범위**로 맞춰집니다.
> 모양(어느 칸이 크고 작은지)은 유지되고, **전체 크기와 위치만** 통일돼요.

In [ ]:
small = torch.tensor([0.02, 0.04, 0.06, 0.08])
huge  = torch.tensor([200.0, 400.0, 600.0, 800.0])

norm = lambda t: (t - t.mean()) / t.var(unbiased=False).sqrt()
print("small 정규화:", r(norm(small)))
print("huge  정규화:", r(norm(huge)))

## 6. LayerNorm은 **어느 축**으로 정규화하나?

`(B, T, C)` 텐서에서 LayerNorm은 **Token 하나하나의 C개 숫자**를 따로 정규화합니다.
- 다른 문장(B)과 섞지 않고
- 다른 Token(T)과도 섞지 않고
- **마지막 축(C)** 만 봅니다 → `dim=-1`

그래서 평균과 분산은 **Token마다 하나씩** 나와요: shape `(B, T, 1)`

In [ ]:
B, T, C = 2, 3, 8
x = torch.randn(B, T, C) * 5 + 3       # 평균 3, 표준편차 5 정도로 일부러 틀어놓기

mean = x.mean(dim=-1, keepdim=True)
var  = x.var(dim=-1, keepdim=True, unbiased=False)

print("x    :", tuple(x.shape))
print("mean :", tuple(mean.shape), " ← Token마다 평균 1개")
print("var  :", tuple(var.shape))

x_hat = (x - mean) / torch.sqrt(var + 1e-5)
print()
print("정규화 후 Token별 평균:\n", x_hat.mean(dim=-1).round(decimals=4) + 0.0)  # + 0.0: -0. 표시 방지
print("정규화 후 Token별 분산:\n", x_hat.var(dim=-1, unbiased=False).round(decimals=3))

> 💡 `keepdim=True`를 쓰는 이유: `(B, T)`가 아니라 `(B, T, 1)`로 남겨야
> `(B, T, C) - (B, T, 1)`처럼 **브로드캐스팅**이 돼서 C개 숫자 모두에서 평균을 뺄 수 있어요.
>
> `+ 1e-5` (eps)는 분산이 0일 때 0으로 나누는 사고를 막는 아주 작은 수입니다.

## 7. 학습 가능한 `weight`(γ)와 `bias`(β)

무조건 평균 0, 분산 1로 맞추면 모델이 "이 Token은 좀 더 크게 두고 싶은데"를 표현할 수 없습니다.
그래서 정규화한 뒤 **다시 조절할 수 있는 손잡이**를 붙입니다.

```text
LayerNorm(x) = γ · (x - 평균) / √(분산 + eps)  +  β
```

- `γ` (weight): 칸마다 얼마나 키울지 — 처음엔 **1**
- `β` (bias): 칸마다 얼마나 옮길지 — 처음엔 **0**
- 둘 다 길이 C, **학습으로 정해짐** (Day 3의 Weight/Bias와 같은 원리)

처음에는 γ=1, β=0이라 그냥 정규화와 똑같고, 필요하면 모델이 스스로 바꿉니다.

In [ ]:
ln = nn.LayerNorm(C)
print("weight (γ):", tuple(ln.weight.shape), r(ln.weight))
print("bias   (β):", tuple(ln.bias.shape), r(ln.bias))
print("파라미터 개수:", sum(p.numel() for p in ln.parameters()), "= 2 × C")

### 7-1. 직접 계산한 것과 `nn.LayerNorm` 비교

In [ ]:
with torch.no_grad():
    ln.weight.copy_(torch.randn(C))      # 학습된 것처럼 아무 값이나 넣어보기
    ln.bias.copy_(torch.randn(C))

mean = x.mean(dim=-1, keepdim=True)
var  = x.var(dim=-1, keepdim=True, unbiased=False)
manual = ln.weight * (x - mean) / torch.sqrt(var + ln.eps) + ln.bias

print("eps:", ln.eps)
print("직접 계산 == nn.LayerNorm ?", torch.allclose(manual, ln(x), atol=1e-6))

> ⚠️ **`unbiased=False` 주의**: `torch.var()`는 기본으로 `N-1`로 나누는 "표본 분산"을 계산합니다.
> LayerNorm은 `N`으로 나누는 분산을 씁니다. 5절에서 손으로 계산한 것처럼요.

In [ ]:
v = torch.tensor([2.0, 4.0, 6.0, 8.0])
print("var(unbiased=True)  → N-1로 나눔:", round(v.var(unbiased=True).item(), 3))
print("var(unbiased=False) → N으로 나눔 :", round(v.var(unbiased=False).item(), 3), " ← LayerNorm은 이것")

## 8. 실험: LayerNorm이 없으면 숫자가 점점 커진다

Residual Stream은 층마다 값을 **더하기만** 하니까, 층이 깊어질수록 숫자가 계속 커집니다.
30층짜리 Residual 스택에서 **각 층(f)에 들어가는 입력의 크기(표준편차)** 를 재봅시다.
- **LN 없음**: `x = x + f(x)` → f는 점점 커지는 x를 그대로 받음
- **LN 있음 (Pre-LN)**: `x = x + f(LN(x))` → f는 항상 정규화된 값을 받음

In [ ]:
depth, C = 30, 64
torch.manual_seed(0)
fs  = [nn.Linear(C, C) for _ in range(depth)]
lns = [nn.LayerNorm(C) for _ in range(depth)]

def run(use_ln):
    torch.manual_seed(1)
    x = torch.randn(32, C)
    stream_std, input_std = [], []
    with torch.no_grad():
        for f, ln in zip(fs, lns):
            inp = ln(x) if use_ln else x
            input_std.append(inp.std().item())
            x = x + f(inp)
            stream_std.append(x.std().item())
    return stream_std, input_std

s_no, in_no = run(use_ln=False)
s_ln, in_ln = run(use_ln=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(range(1, depth + 1), in_no, "o-", label="LN 없음")
axes[0].plot(range(1, depth + 1), in_ln, "o-", label="LN 있음")
axes[0].set_title("각 층 f에 들어가는 입력의 표준편차", fontsize=12)
axes[1].plot(range(1, depth + 1), s_no, "o-", label="LN 없음")
axes[1].plot(range(1, depth + 1), s_ln, "o-", label="LN 있음")
axes[1].set_title("Residual Stream의 표준편차", fontsize=12)
for ax in axes:
    ax.set_xlabel("층 번호")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"30층 입력 표준편차 → LN 없음: {in_no[-1]:.1f},  LN 있음: {in_ln[-1]:.2f}")

- **LN 없음**: 층마다 입력이 점점 커져서 30층에서는 처음보다 수십 배가 됩니다. 이러면 Softmax가 한쪽으로 쏠리고(Day 13), 학습이 불안정해져요.
- **LN 있음**: Stream 자체는 조금씩 커지지만, **각 층이 받는 입력은 항상 표준편차 1**로 일정합니다.

Day 13에서 √dk로 나눈 것도, 오늘 LayerNorm도 결국 같은 목표예요.
> **숫자의 크기를 1 근처로 유지해서 학습을 안정시키자.**

## 9. LLM과의 연결

GPT의 Transformer Block 안에서 Residual과 LayerNorm은 이렇게 쓰입니다. (nanoGPT, GPT-2 방식)

```python
class Block(nn.Module):
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # LN → Attention → Residual
        x = x + self.mlp(self.ln_2(x))    # LN → FFN       → Residual
        return x
```

- LayerNorm을 **f 앞에** 두는 방식을 **Pre-LN**이라고 합니다. 오늘 8절 실험이 바로 이 구조예요.
- 커리큘럼 4-1의 그림처럼 Residual **뒤에** 두는 방식은 원래 Transformer 논문의 **Post-LN**입니다.
- 둘의 차이와 왜 GPT가 Pre-LN을 쓰는지는 **Day 15**에서 자세히 비교합니다.
- 그리고 마지막 Block 뒤, `lm_head` 앞에 **LayerNorm이 하나 더** 있어요. (커리큘럼 6-9의 `ln_f`)

## 10. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** `v = [1, 3, 5, 7, 9]`의 평균과 분산(N으로 나누는 방식)을 **손으로** 계산하세요.

In [ ]:
v = torch.tensor([1.0, 3.0, 5.0, 7.0, 9.0])

my_mean = "?"   # TODO: 숫자
my_var  = "?"   # TODO: 숫자

assert isinstance(my_mean, (int, float)) and abs(my_mean - v.mean().item()) < 1e-6, "평균을 다시 계산해보세요"
assert isinstance(my_var, (int, float)) and abs(my_var - v.var(unbiased=False).item()) < 1e-6, "분산을 다시 계산해보세요 (N으로 나누기)"
print("Q1 정답!")

**Q2.** `nn.LayerNorm`을 쓰지 않고 LayerNorm을 직접 구현하세요.
- 마지막 축(C) 기준
- `weight`(γ), `bias`(β), `eps`까지 반영

In [ ]:
def my_layernorm(x, weight, bias, eps=1e-5):
    return None  # TODO

ln = nn.LayerNorm(16)
with torch.no_grad():
    ln.weight.copy_(torch.randn(16))
    ln.bias.copy_(torch.randn(16))

x = torch.randn(2, 5, 16) * 3 + 1
out = my_layernorm(x, ln.weight, ln.bias, ln.eps)
assert out is not None, "함수를 완성해주세요"
assert out.shape == x.shape, "출력 shape가 입력과 달라요"
assert torch.allclose(out, ln(x), atol=1e-5), "nn.LayerNorm과 결과가 달라요. 축과 분산 계산 방식을 확인해보세요"
print("Q2 정답!")

**Q3.** `x`의 shape가 `(4, 10, 64)`일 때, LayerNorm이 평균을 구하는 축(`dim`)과
`keepdim=True`로 구한 평균의 shape를 적으세요.

In [ ]:
x = torch.randn(4, 10, 64)

my_dim        = "?"   # TODO: 정수 (음수 인덱스도 OK)
my_mean_shape = "?"   # TODO: 튜플

assert my_dim in (2, -1), "LayerNorm은 어느 축의 숫자들끼리 정규화하나요?"
assert my_mean_shape == tuple(x.mean(dim=-1, keepdim=True).shape), "keepdim=True면 그 축은 어떻게 남나요?"
print("Q3 정답!")

**Q4.** `f(x) = x³` 일 때, `x = 2`에서 아래 두 값의 Gradient를 예측하세요.
- `plain = f(x)`
- `resid = x + f(x)`

힌트: `x³`을 미분하면 `3x²`

In [ ]:
grad_plain = "?"   # TODO
grad_resid = "?"   # TODO

x = torch.tensor(2.0, requires_grad=True)
(x ** 3).backward()
true_plain = x.grad.item()

x = torch.tensor(2.0, requires_grad=True)
(x + x ** 3).backward()
true_resid = x.grad.item()

assert grad_plain == true_plain, "f'(x)를 x = 2에서 계산해보세요"
assert grad_resid == true_resid, "x + f(x)를 미분하면 무엇이 더해지나요?"
print("Q4 정답!")

## 11. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Residual Connection은 무엇이고, `out = f(x)`와 무엇이 다른가?
2. Residual이 있으면 왜 깊은 모델에서도 Gradient가 앞쪽 층까지 잘 전달되는가?
3. `x + f(x)`를 하려면 f의 출력 shape는 어떤 조건을 만족해야 하는가? FFN이 `C → 4C → C`인 이유와 연결해보세요.
4. LayerNorm은 `(B, T, C)`에서 어느 축으로 정규화하는가? 평균의 shape는?
5. LayerNorm에 학습 가능한 γ, β가 있는 이유는?
6. LayerNorm은 왜 필요한가? Day 13의 √dk와 어떤 공통점이 있는가?

**내 답:**

1.
2.
3.
4.
5.
6.